# SAEBench: our SAE vs. the published baseline

This notebook analyses the SAEBench eval results produced by
`experiments/eval_saebench.sh` for **one SAE config** and compares them to the
**authors' published numbers** for the matching SAE Bench baseline — the config
you flagged on Neuronpedia,
[`gemma-2-2b/12-sae_bench_0125-batch_topk-res-64k__trainer_2`](https://www.neuronpedia.org/gemma-2-2b/12-sae_bench_0125-batch_topk-res-64k__trainer_2).

The authors' results come from the HuggingFace dataset
[`adamkarvonen/sae_bench_results_0125`](https://huggingface.co/datasets/adamkarvonen/sae_bench_results_0125),
which is the same data Neuronpedia surfaces.

We train **two** SAEs with the **exact trainer_2 recipe** (gemma-2-2b, layer 12
resid_post, 65k width, BatchTopK, **k=80**, **500M tokens**, lr 3e-4, aux 1/32,
threshold beta 0.999, **seed 0** — verified against the published `sae_cfg_dict`):
a **BF16** baseline and an **FP8 TE** run (transformer_engine te.Linear under
fp8_autocast). Identical hyperparameters; only the encoder/decoder GEMM precision
differs. We compare both to the authors' published `trainer_2` numbers.

**Trainer ↔ our member** (65k BatchTopK, mapped by the achieved L0 in the
authors' core results):

| trainer | author L0 | our member |
|---|---|---|
| trainer_0 | ~21  | `w65536_k20`  |
| trainer_1 | ~42  | `w65536_k40`  |
| **trainer_2** | **~84** | **`w65536_k80`** ← the config you sent |
| trainer_3 | ~169 | `w65536_k160` |
| trainer_4 | ~339 | `w65536_k320` |
| trainer_5 | ~696 | `w65536_k640` |

If you evaluated **multiple checkpoints** (`CHECKPOINTS=all`), the last section
plots each metric across training steps, with the author's final value drawn as
a reference line. (The authors only published a *final* checkpoint for
BatchTopK — their per-checkpoint series exist only for the TopK/Standard
variants — so the reference is a single horizontal line.)

## Runtime notes
- Run in the same ROCm container you train/eval in (`pip install -r requirements.txt`).
- This notebook is **read-only + a small HF download**: it does not run the base
  model, so it's cheap. Run `experiments/eval_saebench.sh` first to produce the
  results it reads.
- Downloading the authors' result JSONs needs internet + an HF token for gated
  access is **not** required (the results dataset is public).

## 1. Parameters

In [ ]:
from pathlib import Path

# --- Our results: two runs of the SAME config, FP8 TE vs BF16 ----------------
# Both are trained with byte-for-byte identical hyperparameters matching the
# Neuronpedia/SAEBench trainer_2 config (train_saebench_replication.py, default
# 500M-token recipe: --model gemma --widths 65536 --ks 80 --seed 0); the only
# difference is the encoder/decoder GEMM precision. The BF16 run is the default
# (--no fp8 flags); the FP8 TE run adds --fp8-te (transformer_engine te.Linear
# under fp8_autocast, hybrid+delayed). eval_saebench.sh writes
# <RUN_DIR>/saebench_eval/<member>/eval_results/, which is what we read here.
MEMBER = "w65536_k80"  # the 65k batch_topk trainer_2 config you sent (L0~84 -> k=80)
RESULTS_ROOT = Path("/wekafs/smerrill/efficient_sae/experiments/results")

# label -> the run's saebench_eval/<member> dir. Missing runs are skipped, so this
# still works before one of the evals has been produced.
OURS_RUNS = {
    "FP8 TE (ours)": RESULTS_ROOT / "saebench_gemma_fp8te" / "saebench_eval" / MEMBER,
    "BF16 (ours)":   RESULTS_ROOT / "saebench_gemma"       / "saebench_eval" / MEMBER,
}
# Back-compat alias used by a couple of cells (the FP8 TE run is the headline series).
OUR_OUTPUT_DIR = OURS_RUNS["FP8 TE (ours)"]

# Which eval types to analyse. We run only core + sparse_probing across all
# checkpoints (the GPU-bound, fast, headline metrics — exactly what Neuronpedia
# shows: L0 / loss-recovered / sparse-probing). The expensive CPU-bound evals
# (absorption / scr / tpp) are too slow at 65k width to sweep over checkpoints,
# so they're intentionally excluded here. Add them to this list if you ran them
# (e.g. once on the final checkpoint) and want them in the comparison.
EVAL_TYPES = ["core", "sparse_probing"]

# --- Authors' published baseline (HuggingFace) -------------------------------
AUTHOR_RESULTS_REPO = "adamkarvonen/sae_bench_results_0125"
# The 65k BatchTopK gemma-2-2b SAEs are published under the date-0108 release
# (the "0125" in the Neuronpedia id is the upload batch, not the results dir).
AUTHOR_RELEASE = "saebench_gemma-2-2b_width-2pow16_date-0108"
AUTHOR_ARCH = "BatchTopK"
AUTHOR_MODEL_TOKEN = "gemma-2-2b__0108"
AUTHOR_LAYER = 12

# k -> SAEBench trainer index, by achieved L0 (65k BatchTopK suite).
K_TO_TRAINER = {20: 0, 40: 1, 80: 2, 160: 3, 320: 4, 640: 5}

print(f"member       = {MEMBER}")
print(f"eval types   = {EVAL_TYPES}")
print("our runs     :")
for _label, _d in OURS_RUNS.items():
    print(f"   {_label:14s} {'[present]' if (_d / 'eval_results').exists() else '[missing]'} "
          f"{_d / 'eval_results'}")
print(f"author repo  = {AUTHOR_RESULTS_REPO}")
print(f"author release = {AUTHOR_RELEASE}")

member       = w65536_k80
eval types   = ['core', 'sparse_probing']
our runs     :
   FP8 (ours)   [missing] /wekafs/smerrill/efficient_sae/experiments/results/saebench_gemma_fp8/saebench_eval/w65536_k80/eval_results
   FP16 (ours)  [present] /wekafs/smerrill/efficient_sae/experiments/results/saebench_gemma/saebench_eval/w65536_k80/eval_results
author repo  = adamkarvonen/sae_bench_results_0125
author release = saebench_gemma-2-2b_width-2pow16_date-0108


## 2. Imports & the metric registry

In [2]:
import glob
import json
import re

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from huggingface_hub import hf_hub_download

pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")


def dig(d: dict, *path, default=None):
    '''Safely walk nested dicts: dig(metrics, "sparsity", "l0").'''
    cur = d
    for p in path:
        if not isinstance(cur, dict) or p not in cur:
            return default
        cur = cur[p]
    return cur


# Headline metrics per eval type: friendly_name -> (group, key, higher_is_better).
# These are extracted from eval_result_metrics in each result JSON.
METRIC_REGISTRY = {
    "core": {
        "L0":               ("sparsity", "l0", None),
        "CE_loss_score":    ("model_performance_preservation", "ce_loss_score", True),
        "KL_div_score":     ("model_behavior_preservation", "kl_div_score", True),
        "explained_var":    ("reconstruction_quality", "explained_variance", True),
        "frac_alive":       ("misc_metrics", "frac_alive", True),
    },
    "sparse_probing": {
        "probe_top1_acc":   ("sae", "sae_top_1_test_accuracy", True),
        "probe_top2_acc":   ("sae", "sae_top_2_test_accuracy", True),
        "probe_top5_acc":   ("sae", "sae_top_5_test_accuracy", True),
        "probe_full_acc":   ("sae", "sae_test_accuracy", True),
        "llm_top1_acc":     ("llm", "llm_top_1_test_accuracy", True),
    },
    "absorption": {
        "absorption_fraction": ("mean", "mean_absorption_fraction_score", False),
        "full_absorption":     ("mean", "mean_full_absorption_score", False),
        "num_split_features":  ("mean", "mean_num_split_features", False),
    },
    "scr": {
        "scr_thr_10": ("scr_metrics", "scr_metric_threshold_10", True),
        "scr_thr_20": ("scr_metrics", "scr_metric_threshold_20", True),
    },
    "tpp": {
        "tpp_thr_10": ("tpp_metrics", "tpp_threshold_10_total_metric", True),
        "tpp_thr_20": ("tpp_metrics", "tpp_threshold_20_total_metric", True),
    },
    "autointerp": {
        "autointerp_score": ("autointerp", "autointerp_score", True),
    },
    "ravel": {
        "disentanglement": ("ravel", "disentanglement_score", True),
        "cause":           ("ravel", "cause_score", True),
        "isolation":       ("ravel", "isolation_score", True),
    },
    "unlearning": {
        "unlearning_score": ("unlearning", "unlearning_score", True),
    },
}

HIGHER_IS_BETTER = {
    name: hib
    for et in METRIC_REGISTRY.values()
    for name, (_, _, hib) in et.items()
}


def extract_metrics(eval_type: str, result_json: dict) -> dict:
    '''Pull the headline metrics for one eval type out of a result JSON.'''
    m = result_json.get("eval_result_metrics", {})
    out = {}
    for name, (grp, key, _) in METRIC_REGISTRY.get(eval_type, {}).items():
        out[name] = dig(m, grp, key)
    return out

## 3. Load OUR eval results (all checkpoints)

In [3]:
# eval_results/<eval_type>/<tag>_<member>_step_<STEP>_custom_sae_eval_results.json
step_re = re.compile(r"_step_(\d+)_")


def load_eval_dir(run_dir: Path, member: str, eval_types) -> pd.DataFrame:
    """Long df (eval_type, metric, step, value) for one run's eval_results/ dir.

    Returns an empty df if the run hasn't been evaluated yet."""
    eval_dir = run_dir / "eval_results"
    rows = []
    if eval_dir.exists():
        for path in sorted(glob.glob(str(eval_dir / "*" / "*_eval_results.json"))):
            p = Path(path)
            eval_type = p.parent.name
            if eval_type not in eval_types or member not in p.name:
                continue
            sm = step_re.search(p.name)
            if sm is None:
                continue
            data = json.loads(p.read_text())
            for metric, value in extract_metrics(eval_type, data).items():
                rows.append({"eval_type": eval_type, "metric": metric,
                             "step": int(sm.group(1)), "value": value})
    return pd.DataFrame(rows, columns=["eval_type", "metric", "step", "value"])


# Load every run that has results. ours_by_prec maps label -> long df.
ours_by_prec = {}
for label, d in OURS_RUNS.items():
    df = load_eval_dir(d, MEMBER, EVAL_TYPES)
    if df.empty:
        print(f"  {label:12s} [skipped — no eval results under {d / 'eval_results'}]")
    else:
        print(f"  {label:12s} steps={sorted(df['step'].unique())}  "
              f"evals={sorted(df['eval_type'].unique())}")
        ours_by_prec[label] = df

assert ours_by_prec, (
    "No eval results found for ANY run. Run experiments/eval_saebench.sh first "
    f"(e.g. RUN_DIR=results/saebench_gemma_fp8te MEMBER={MEMBER} ./eval_saebench.sh)."
)

# Headline / "primary" series: prefer FP8 TE (the new result), else whatever loaded.
PRIMARY = "FP8 TE (ours)" if "FP8 TE (ours)" in ours_by_prec else next(iter(ours_by_prec))
ours_long = ours_by_prec[PRIMARY]
steps = sorted(ours_long["step"].unique())
final_step = max(steps)
print(f"\nprimary series = {PRIMARY}  (final step {final_step:,}, "
      f"{len(steps)} checkpoint(s))")
ours_long.head(20)

  FP8 (ours)   [skipped — no eval results under /wekafs/smerrill/efficient_sae/experiments/results/saebench_gemma_fp8/saebench_eval/w65536_k80/eval_results]


  FP16 (ours)  steps=[23810048, 47620096, 499998720]  evals=['core', 'sparse_probing']

primary series = FP16 (ours)  (final step 499,998,720, 3 checkpoint(s))


,eval_type,metric,step,value
0,core,L0,23810048,86.0226
1,core,CE_loss_score,23810048,0.9852
2,core,KL_div_score,23810048,0.9866
3,core,explained_var,23810048,0.8633
4,core,frac_alive,23810048,0.9529
5,core,L0,47620096,84.6856
6,core,CE_loss_score,47620096,0.9868
7,core,KL_div_score,47620096,0.9872
8,core,explained_var,47620096,0.8672
9,core,frac_alive,47620096,0.9418


## 4. Fetch the AUTHORS' published baseline

In [4]:
trainer = K_TO_TRAINER[int(MEMBER.split("_k")[1])]
print(f"{MEMBER}  ->  {AUTHOR_RELEASE}  trainer_{trainer}")

author_rows = []
author_missing = []
for eval_type in sorted(ours_long["eval_type"].unique()):
    fname = (
        f"{eval_type}/{AUTHOR_RELEASE}/{AUTHOR_RELEASE}_{AUTHOR_ARCH}_"
        f"{AUTHOR_MODEL_TOKEN}_resid_post_layer_{AUTHOR_LAYER}_trainer_{trainer}"
        f"_eval_results.json"
    )
    try:
        local = hf_hub_download(AUTHOR_RESULTS_REPO, fname, repo_type="dataset")
        data = json.loads(Path(local).read_text())
    except Exception as e:  # noqa: BLE001
        author_missing.append((eval_type, str(e).splitlines()[0][:80]))
        continue
    for metric, value in extract_metrics(eval_type, data).items():
        author_rows.append({"eval_type": eval_type, "metric": metric, "value": value})

authors = pd.DataFrame(author_rows)
if author_missing:
    print("Could not fetch author results for:", author_missing)
print(f"\nFetched authors' trainer_{trainer} metrics for: "
      f"{sorted(authors['eval_type'].unique()) if not authors.empty else 'none'}")
authors

w65536_k80  ->  saebench_gemma-2-2b_width-2pow16_date-0108  trainer_2


(…)ost_layer_12_trainer_2_eval_results.json: 0.00B [00:00, ?B/s]

(…)ost_layer_12_trainer_2_eval_results.json: 0.00B [00:00, ?B/s]


Fetched authors' trainer_2 metrics for: ['core', 'sparse_probing']


,eval_type,metric,value
0,core,L0,84.3313
1,core,CE_loss_score,0.9918
2,core,KL_div_score,0.9912
3,core,explained_var,0.7734
4,core,frac_alive,0.8703
5,sparse_probing,probe_top1_acc,0.7393
6,sparse_probing,probe_top2_acc,0.7916
7,sparse_probing,probe_top5_acc,0.8569
8,sparse_probing,probe_full_acc,0.9556
9,sparse_probing,llm_top1_acc,0.6535


## 5. Head-to-head comparison (FP8 TE vs BF16 vs authors, final checkpoint)

One column per source: our **FP8 TE** run, our **BF16** run, and the authors'
`trainer_2` baseline. The headline question is **FP8 TE vs our own BF16** (same
recipe, only the GEMM precision changes): `fp8te−bf16` is that delta and the arrow
shows whether FP8 TE moved in the *better* direction for that metric (↑/↓ = better,
— = ~equal). L0 has no "better" direction — it's the achieved sparsity (should
land near the author's ~84 for trainer_2). Runs that haven't been evaluated yet
are simply omitted.

In [5]:
# Final-checkpoint value for each of our runs, plus the authors' baseline.
auth_idx = authors.set_index(["eval_type", "metric"])["value"] if not authors.empty else pd.Series(dtype=float)
final_vals = {
    label: df[df["step"] == df["step"].max()].set_index(["eval_type", "metric"])["value"]
    for label, df in ours_by_prec.items()
}

all_keys = sorted(set().union(*[set(s.index) for s in final_vals.values()],
                              set(auth_idx.index)))
prec_cols = list(ours_by_prec)  # e.g. ["FP8 TE (ours)", "BF16 (ours)"]

cmp_rows = []
for (eval_type, metric) in all_keys:
    row = {"eval_type": eval_type, "metric": metric}
    for label in prec_cols:
        row[label] = final_vals[label].get((eval_type, metric))
    row["authors"] = auth_idx.get((eval_type, metric))
    hib = HIGHER_IS_BETTER.get(metric)
    # Headline question: FP8 TE vs BF16 (does dropping to fp8 cost anything?).
    if {"FP8 TE (ours)", "BF16 (ours)"} <= set(prec_cols):
        f8, f16 = row["FP8 TE (ours)"], row["BF16 (ours)"]
        d = (f8 - f16) if (f8 is not None and f16 is not None) else None
        row["fp8te−bf16"] = d
        v = "—"
        if d is not None and hib is not None and abs(d) > 1e-9:
            v = "↑ better" if ((d > 0) == hib) else "↓ worse"
        row["fp8te_vs_bf16"] = v
    cmp_rows.append(row)

comparison = pd.DataFrame(cmp_rows).set_index(["eval_type", "metric"])
print(f"Final checkpoints  ({', '.join(prec_cols)})  vs  authors' trainer_{trainer}")
print("(fp8te_vs_bf16 arrow = is FP8 TE better/worse than our BF16 for that metric)\n")
comparison

Final checkpoints  (FP16 (ours))  vs  authors' trainer_2
(fp8_vs_fp16 arrow = is FP8 better/worse than our FP16 for that metric)



FP16 (ours)  authors
eval_type      metric                              
core           CE_loss_score        0.9901   0.9918
               KL_div_score         0.9902   0.9912
               L0                  84.9212  84.3313
               explained_var        0.8828   0.7734
               frac_alive           0.9906   0.8703
sparse_probing llm_top1_acc         0.6529   0.6535
               probe_full_acc       0.9554   0.9556
               probe_top1_acc       0.7005   0.7393
               probe_top2_acc       0.7446   0.7916
               probe_top5_acc       0.8522   0.8569

In [6]:
# Grouped bar: every source (FP8 TE, BF16, authors) per shared metric (L0 excluded — different scale).
source_cols = [c for c in [*prec_cols, "authors"] if c in comparison.columns]
plot_df = comparison.reset_index()
plot_df = plot_df[plot_df["metric"] != "L0"]
melted = plot_df.melt(id_vars=["eval_type", "metric"], value_vars=source_cols,
                      var_name="source", value_name="value").dropna(subset=["value"])
if not melted.empty:
    melted["label"] = melted["eval_type"] + " · " + melted["metric"]
    fig = px.bar(melted, x="label", y="value", color="source", barmode="group",
                 title=f"{MEMBER}: FP8 TE vs BF16 (ours) vs authors' trainer_{trainer}",
                 text_auto=".3f")
    fig.update_layout(height=470, width=1050, xaxis_title="", yaxis_title="",
                      xaxis_tickangle=-35, legend_title="")
    fig.show()
else:
    print("No overlapping metrics to plot (did the author fetch / eval loads succeed?).")

## 6. Metrics across training checkpoints

Only meaningful if you ran `CHECKPOINTS=all` (or a step list). Each metric is
plotted against the training step with **one line per precision** (FP8 / FP16),
and the dashed line is the author's final value (trainer_2). This shows whether
FP8 and FP16 track each other throughout training and where each lands vs the
published baseline. (The FP8 run uses `--n-checkpoints 20`; eval it with
`CHECKPOINTS=all RUN_DIR=results/saebench_gemma_fp8 ./eval_saebench.sh`.)

In [7]:
max_ckpts = max(df["step"].nunique() for df in ours_by_prec.values())
if max_ckpts <= 1:
    print("Only one checkpoint per run evaluated — re-run eval_saebench.sh with "
          "CHECKPOINTS=all to populate this section.\n"
          "(Requires a training run WITH intermediate checkpoints, e.g.\n"
          "   N_CHECKPOINTS=20 WIDTHS=65536 KS=80 ./train_saebench.sh\n"
          " then CHECKPOINTS=all RUN_DIR=results/saebench_gemma_fp8te ./eval_saebench.sh )")
else:
    # One trace per precision (FP8 TE, BF16) + the authors' final value as a dashed line.
    metrics = sorted(set().union(*[
        set(df[["eval_type", "metric"]].itertuples(index=False, name=None))
        for df in ours_by_prec.values()]))
    for eval_type, metric in metrics:
        fig = go.Figure()
        plotted = False
        for label, df in ours_by_prec.items():
            sub = (df[(df.eval_type == eval_type) & (df.metric == metric)]
                   .dropna(subset=["value"]).sort_values("step"))
            if sub.empty:
                continue
            fig.add_trace(go.Scatter(x=sub["step"], y=sub["value"],
                                     mode="lines+markers", name=label))
            plotted = True
        if not plotted:
            continue
        auth_val = auth_idx.get((eval_type, metric)) if not authors.empty else None
        if auth_val is not None:
            fig.add_hline(y=auth_val, line_dash="dash", line_color="firebrick",
                          annotation_text=f"authors trainer_{trainer} = {auth_val:.4f}",
                          annotation_position="top left")
        hib = HIGHER_IS_BETTER.get(metric)
        better = {True: " (higher better)", False: " (lower better)"}.get(hib, "")
        fig.update_layout(title=f"{eval_type} · {metric}{better}",
                          height=340, width=760, legend_title="",
                          xaxis_title="training step", yaxis_title=metric)
        fig.show()

## Notes
- **What "comparable" means:** `eval_saebench.py` uses SAEBench's own
  `run_all_evals_custom_saes.run_evals`, i.e. the *same* eval configs the authors
  used (core: 200 reconstruction / 2000 sparsity-variance batches on
  `Skylion007/openwebtext`, ctx 128, bf16). Remaining differences are training
  (we reimplement the recipe in SAELens, the authors used `dictionary_learning`),
  so small gaps are expected; large gaps are worth investigating.
- **Other configs:** point `MEMBER` at any `w<width>_k<k>` you evaluated and the
  trainer mapping + author release auto-adjust (re-derive `AUTHOR_RELEASE`'s
  `width-2pow..` if you change width).
- **More evals:** the comparison automatically covers whichever eval types you
  ran (`absorption`, `scr`, `tpp`, `autointerp`, `ravel`, `unlearning` are all in
  the registry). Run them via `EVALS=all ./eval_saebench.sh`.